In [1]:
import pandas as pd
import numpy as np
import os

# Load main dataset
df = pd.read_csv("Data/NHANES_age_prediction.csv")

# Load body measurements
df_response = pd.read_csv("Data/response_clean.csv", low_memory=False)
primary_bmx = ['SEQN'] + [c for c in df_response.columns 
               if c.startswith('BMX') and not c[-1].isdigit()]

# Merge on SEQN
df_merged = df.merge(df_response[primary_bmx], on='SEQN', how='left')

# Clean up duplicate BMI column
df_merged.drop(columns=['BMXBMI_y'], inplace=True, errors='ignore')
df_merged.rename(columns={'BMXBMI_x': 'BMXBMI'}, inplace=True, errors='ignore')

pd.set_option('display.max_columns', None)
print("Shape:", df_merged.shape)
df_merged.head()


Shape: (2278, 23)


,SEQN,age_group,RIDAGEYR,RIAGENDR,PAQ605,BMXBMI,LBXGLU,DIQ010,LBXGLT,LBXIN,BMXARMC,BMXARML,BMXCALF,BMXHEAD,BMXHIP,BMXHT,BMXLEG,BMXRECUM,BMXSUB,BMXTHICR,BMXTRI,BMXWAIST,BMXWT
0,73564.0,Adult,61.0,2.0,2.0,35.7,110.0,2.0,150.0,14.91,38.0,39.3,NaN,NaN,NaN,161.8,37.1,NaN,NaN,NaN,NaN,110.8,93.4
1,73568.0,Adult,26.0,2.0,2.0,20.3,89.0,2.0,80.0,3.85,25.8,32.6,NaN,NaN,NaN,152.5,34.4,NaN,NaN,NaN,NaN,73.7,47.1
2,73576.0,Adult,16.0,1.0,2.0,23.2,89.0,2.0,68.0,6.14,30.7,37.0,NaN,NaN,NaN,170.4,42.0,NaN,NaN,NaN,NaN,74.2,67.3
3,73577.0,Adult,32.0,1.0,2.0,28.9,104.0,2.0,84.0,16.15,34.5,37.0,NaN,NaN,NaN,166.2,36.5,NaN,NaN,NaN,NaN,100.0,79.7
4,73580.0,Adult,38.0,2.0,1.0,35.9,103.0,2.0,81.0,10.92,43.0,39.5,NaN,NaN,NaN,161.4,40.0,NaN,NaN,NaN,NaN,107.4,93.5


In [4]:
import os
os.makedirs('Models', exist_ok=True)
joblib.dump(rf_gender, 'Models/gender_model.pkl')
joblib.dump(rf_age, 'Models/age_model.pkl')
joblib.dump(scaler, 'Models/scaler.pkl')
joblib.dump(X.columns.tolist(), 'Models/feature_names.pkl')
print("✅ Models saved!")

✅ Models saved!


In [5]:
import pickle
pickle.dump(rf_gender, open('Models/gender_model.pkl', 'wb'))
pickle.dump(rf_age, open('Models/age_model.pkl', 'wb'))
pickle.dump(scaler, open('Models/scaler.pkl', 'wb'))
print("✅ Done!")


✅ Done!


In [2]:
import pandas as pd
import numpy as np

# Load main dataset
df = pd.read_csv("Data/NHANES_age_prediction.csv")

# Load body measurements
df_response = pd.read_csv("Data/response_clean.csv", low_memory=False)
primary_bmx = ['SEQN'] + [c for c in df_response.columns 
               if c.startswith('BMX') and not c[-1].isdigit()]

# Merge on SEQN
df_merged = df.merge(df_response[primary_bmx], on='SEQN', how='left')
df_merged.drop(columns=['BMXBMI_y'], inplace=True, errors='ignore')
df_merged.rename(columns={'BMXBMI_x': 'BMXBMI'}, inplace=True, errors='ignore')

print("Shape:", df_merged.shape)
print("Min age:", df_merged['RIDAGEYR'].min())
print("Max age:", df_merged['RIDAGEYR'].max())
print("\nAge distribution:")
print(df_merged['RIDAGEYR'].value_counts().sort_index().head(30))

Shape: (2278, 23)
Min age: 12.0
Max age: 80.0

Age distribution:
RIDAGEYR
12.0    58
13.0    49
14.0    60
15.0    48
16.0    70
17.0    43
18.0    70
19.0    40
20.0    32
21.0    33
22.0    31
23.0    33
24.0    31
25.0    35
26.0    40
27.0    33
28.0    29
29.0    25
30.0    34
31.0    36
32.0    35
33.0    31
34.0    36
35.0    25
36.0    45
37.0    26
38.0    31
39.0    28
40.0    41
41.0    37
Name: count, dtype: int64


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
import pickle

# Drop 100% missing columns and SEQN
cols_to_drop = ['BMXCALF', 'BMXHIP', 'BMXHEAD', 'BMXTRI', 'BMXTHICR', 'BMXRECUM', 'BMXSUB', 'SEQN']
df_clean = df_merged.drop(columns=cols_to_drop)

# Fill missing values
num_cols = df_clean.select_dtypes(include='number').columns
df_clean[num_cols] = SimpleImputer(strategy='median').fit_transform(df_clean[num_cols])

# New age groups
def assign_age_group(age):
    if age <= 19:   return 'Teen'
    elif age <= 59: return 'Adult'
    else:           return 'Senior'

df_clean['age_group_new'] = df_clean['RIDAGEYR'].apply(assign_age_group)

print("New age group distribution:")
print(df_clean['age_group_new'].value_counts())

New age group distribution:
age_group_new
Adult     1314
Senior     526
Teen       438
Name: count, dtype: int64


In [4]:
from sklearn.preprocessing import LabelEncoder

# Encode targets
le_age = LabelEncoder()
df_clean['age_group_enc'] = le_age.fit_transform(df_clean['age_group_new'])
df_clean['gender_enc'] = df_clean['RIAGENDR'].astype(int)

print("Age group encoding:", dict(zip(le_age.classes_, le_age.transform(le_age.classes_))))

# Features
X = df_clean.drop(columns=['age_group', 'RIAGENDR', 'age_group_enc', 'gender_enc', 'RIDAGEYR', 'age_group_new'])
y_gender = df_clean['gender_enc']
y_age = df_clean['age_group_enc']

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split
X_train, X_test, yg_train, yg_test, ya_train, ya_test = train_test_split(
    X_scaled, y_gender, y_age, test_size=0.2, random_state=42, stratify=y_gender)

# Gender model
rf_gender = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_gender.fit(X_train, yg_train)
yg_pred = rf_gender.predict(X_test)
print("\nGENDER ACCURACY:", accuracy_score(yg_test, yg_pred))

# Age model with SMOTE
sm = SMOTE(random_state=42)
X_train_sm, ya_train_sm = sm.fit_resample(X_train, ya_train)
rf_age = RandomForestClassifier(n_estimators=100, random_state=42)
rf_age.fit(X_train_sm, ya_train_sm)
ya_pred = rf_age.predict(X_test)
print("AGE GROUP ACCURACY:", accuracy_score(ya_test, ya_pred))
print(classification_report(ya_test, ya_pred, target_names=le_age.classes_))

# Save models
pickle.dump(rf_gender, open('Models/gender_model.pkl', 'wb'))
pickle.dump(rf_age, open('Models/age_model.pkl', 'wb'))
pickle.dump(scaler, open('Models/scaler.pkl', 'wb'))
pickle.dump(le_age, open('Models/age_label_encoder.pkl', 'wb'))
print("\n✅ Models saved!")


Age group encoding: {'Adult': np.int64(0), 'Senior': np.int64(1), 'Teen': np.int64(2)}

GENDER ACCURACY: 0.8640350877192983
AGE GROUP ACCURACY: 0.6381578947368421
              precision    recall  f1-score   support

       Adult       0.70      0.68      0.69       262
      Senior       0.47      0.49      0.48       103
        Teen       0.67      0.70      0.68        91

    accuracy                           0.64       456
   macro avg       0.61      0.62      0.62       456
weighted avg       0.64      0.64      0.64       456


✅ Models saved!
